In [ ]:
from calculation_functions import *
from cleaning_functions import *
from lookup_tables import fuel_prices_df
import math
import numpy.random as npr
import pandas as pd

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
seed = 290923
npr.seed(seed)

## `address_profiling_2023_09_15_1430.csv`:

In [4]:
df = pd.read_csv(
    's3://bays-rdsap-data/raw_data/address_profiling_2023_09_15_1430.csv',
    low_memory=False)
df.shape

(65077, 45)

### Dropping the empty `HeatingCost(£/yr)` column:

In [ ]:
df = df.drop(columns='HeatingCost(£/yr)', inplace=True)

### Some data manipulation to make the data consistent:

In [6]:
df.loc[df.MainFuelType.isna(), 'MainFuelType'] = 'Other'
df.loc[df.MainFuelType == 'Invalid', 'MainFuelType'] = 'Other'
df.loc[df.MainFuelType == 'NoSystem', 'MainFuelType'] = 'Other'

### To calculate the heating energy, the average heating energy per square meter per annum has been used. [[source]](https://www.ovoenergy.com/guides/energy-guides/how-much-heating-energy-do-you-use)

average heating energy per square meter per annum (UK) = 133 kWh/m2a

This implies that `total heating energy` = `average heating energy per square meter per annum` x `TotalFloorArea`

### Total Heating Energy Cost:

Once we have the total energy, the below lookup table was used to calculate the annual heating energy cost. [Source: Page 225](https://files.bregroup.com/SAP/SAP-2012_9-92.pdf)

In [7]:
fuel_prices_df

,standing_cost,unit_price
MainsGas,120.0,0.0348
Electricity,54.0,0.1319
Oil,0.0,0.0544
LPG,70.0,0.0760
DualFuel,0.0,0.0399
WoodLogs,0.0,0.0423
Coal,0.0,0.0367
Biomass,0.0,0.4700
Anthracite,0.0,0.0364
SmokelessCoal,0.0,0.0367


In [ ]:
def calculate_energy_used(
    s: pd.Series,
    avg_kwh_per_m2_per_annum: float = 133,
):
    """
    Calculate the heating energy used by using the average heating energy per square meter per annum.
    source: # https://www.ovoenergy.com/guides/energy-guides/how-much-heating-energy-do-you-use
    """
    return s.TotalFloorArea * avg_kwh_per_m2_per_annum


def calculate_heating_cost(
    s: pd.Series,
    fuel_prices_df: pd.DataFrame,
):
    """
    Calculates the Heating cost per annum.
    Source for prices: https://files.bregroup.com/SAP/SAP-2012_9-92.pdf (page 225)
    """
    heating_energy_used = calculate_energy_used(s)
    heating_fuel = s.MainFuelType
    fuel_prices = fuel_prices_df.loc[heating_fuel]
    heating_cost = fuel_prices['standing_cost'] + fuel_prices['unit_price'] * heating_energy_used
    return heating_cost

In [9]:
df['HeatingCost(£/yr)'] = df.apply(lambda row: calculate_heating_cost(row, fuel_prices_df), axis=1)

### Total Energy Cost:

The `WaterHeatingCost` and `LightingCost` are already in the dataset. They are added to the calculated `HeatingCost` to get the `totalEnergyCost`.

In [10]:
df['totalEnergyCost'] = df.apply(lambda row: calculate_total_energy_cost(row), axis=1)

### SAP Score calculation:

To calculate the SAP score, the `Energy Cost Factor` was first calculated. It uses the `totalEnergyCost`, `TotalFloorArea` and an `Energy Cost Deflator` of `0.42` such that:

`Energy Cost Factor` = `(totalEnergyCost * energy_cost_deflator)` / `(totalFloorArea + 45.0)`

Once the `Energy Cost Factor` is obtained, the SAP score is calculated as shown below:

if `Energy Cost Factor` > `3.4`:
    
`SAP Score` = `111 - (110 * log10(Energy Cost Factor))`

otherwise:

`SAP Score` = `110 - 13.96 * Energy Cost Factor`

In [ ]:
def calculate_energy_cost_factor(
    s: pd.Series,
    energy_cost_deflator: float = 0.42
) -> float:
    """
    Calculates the energy cost factor to be used in the SAP calculation.
    """
    total_energy_cost = calculate_total_energy_cost(s)
    total_floor_area = s.TotalFloorArea
    return (total_energy_cost * energy_cost_deflator) / (total_floor_area + 45.0)


def calculate_sap_score(
    s: pd.Series
) -> int:
    """
    Calculates the SAP score.
    """
    energy_cost_factor = calculate_energy_cost_factor(s)
    if energy_cost_factor > 3.4:
        return int(111 - (110 * math.log10(energy_cost_factor)))
    else:
        return int(110 - 13.96 * energy_cost_factor)

In [12]:
df['SAP Score'] = df.apply(lambda row: calculate_sap_score(row), axis=1)

In [13]:
df['SAP Score']

0        89
1        41
2        77
3        91
4        48
         ..
65072    79
65073    80
65074    81
65075    28
65076    73
Name: SAP Score, Length: 65077, dtype: int64

# END OF NOTEBOOK.
___